<a href="https://colab.research.google.com/github/MacKumachin/Clifford-FBSM-SignalControl/blob/main/TableA1Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Completed-results empirical analogue estimation script
=====================================================

Purpose
-------
This is a strict replacement for the earlier empirical analogue script.
It is designed for a *completed robustness-results* submission standard.
Accordingly, it:

1. requires local observed data for every specification;
2. forbids synthetic proxy generation and silent fallbacks;
3. estimates every row of Appendix Table A.1 from script;
4. writes trace files, posterior summaries, daily posterior paths,
   diagnostics, a manifest, and the LaTeX robustness table;
5. raises an error if any required specification cannot be estimated.

Required episode files
----------------------
You should provide three local CSV files:

  (i)  baseline/truss episode CSV
  (ii) placebo March 2022 CSV
  (iii) placebo August 2022 CSV

Each CSV must contain at least the following observed columns, either under
these exact names or under one of the recognised aliases below:

  - date
  - FX stress proxy                 (e.g. usd_per_gbp / DEXUSUK)
  - reset / intervention proxy      (e.g. boe_purchase_total)
  - main gilt-stress proxy
  - main liquidity-stress proxy

The baseline CSV must additionally contain:

  - alternative gilt-stress proxy
  - alternative liquidity-stress proxy

This script deliberately stops if any required observed series is missing.
That behaviour is intentional: a completed-results claim should not depend on
fabricated or substituted data.

Example
-------
python estimate_empirical_analogue_completed.py ¥
    --baseline-csv data/truss_empirical_completed.csv ¥
    --placebo-march-csv data/placebo_march_2022_completed.csv ¥
    --placebo-aug-csv data/placebo_aug_2022_completed.csv ¥
    --outdir output_completed
"""

from __future__ import annotations

import argparse
import hashlib
import json
import logging
import os
import platform
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

# Set PyTensor flags before importing PyMC / PyTensor.
os.environ.setdefault("PYTENSOR_FLAGS", "optimizer=fast_compile")

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

logging.getLogger("pymc").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# -----------------------------------------------------------------------------
# Alias dictionaries
# -----------------------------------------------------------------------------
DATE_ALIASES = [
    "date",
    "Date",
    "DATE",
    "observation_date",
    "ObservationDate",
    "timestamp",
    "day",
]

FX_ALIASES = [
    "usd_per_gbp",
    "USD_per_GBP",
    "gbpusd",
    "GBPUSD",
    "DEXUSUK",
    "fx",
    "exchange_rate",
]

RESET_ALIASES = [
    "boe_purchase_total",
    "boe_purchase",
    "boe_purchases_mn",
    "purchase_mn",
    "amount_mn",
    "reset",
    "reset_proxy",
    "intervention",
    "intervention_proxy",
]

GILT_MAIN_ALIASES = [
    "gilt_yield",
    "gilt_stress",
    "long_gilt_yield",
    "uk10y",
    "uk30y",
    "gilt_proxy",
    "gilt_main",
]

LIQ_MAIN_ALIASES = [
    "bid_ask_spread",
    "liquidity_spread",
    "liq_proxy",
    "liquidity_proxy",
    "liquidity_main",
    "repo_spread",
    "swap_spread",
]

GILT_ALT_ALIASES = [
    "gilt_yield_alt",
    "gilt_stress_alt",
    "uk10y_alt",
    "uk30y_alt",
    "gilt_proxy_alt",
    "gilt_alternative",
    "gilt_alt",
]

LIQ_ALT_ALIASES = [
    "bid_ask_spread_alt",
    "liquidity_spread_alt",
    "liq_proxy_alt",
    "liquidity_proxy_alt",
    "liquidity_alternative",
    "liquidity_alt",
]


# -----------------------------------------------------------------------------
# Dates and specification template
# -----------------------------------------------------------------------------
MB_START = pd.Timestamp("2022-09-23")
MB_END = pd.Timestamp("2022-09-27")
BOE_START = pd.Timestamp("2022-09-28")
BOE_END = pd.Timestamp("2022-10-14")
REV_START = pd.Timestamp("2022-10-15")

# Event-window defaults matching the manuscript's robustness logic.
BASELINE_START = pd.Timestamp("2022-09-20")
BASELINE_END = pd.Timestamp("2022-10-31")
SHORT_START = pd.Timestamp("2022-09-23")
SHORT_END = pd.Timestamp("2022-10-14")
LONG_START = pd.Timestamp("2022-09-13")
LONG_END = pd.Timestamp("2022-10-31")

# Relative placebo template copied from the earlier empirical design.
PLACEBO_MB_SLICE = (3, 8)   # [3,8)
PLACEBO_BOE_SLICE = (8, 25) # [8,25)
PLACEBO_REV_START = 25
PLACEBO_RUPTURE_SLICE = (8, 11)  # [8,11) -> three days


# -----------------------------------------------------------------------------
# Data structures
# -----------------------------------------------------------------------------
@dataclass
class EpisodeFrame:
    name: str
    df: pd.DataFrame
    source_path: Path


@dataclass
class PreparedData:
    spec_name: str
    episode_name: str
    start_date: str
    end_date: str
    state_spec: str
    proxy_set: str
    T: int
    dates: List[str]
    z_fx: np.ndarray
    z_gilt: np.ndarray
    z_liq: np.ndarray
    z_reset: np.ndarray
    d_MB: np.ndarray
    d_BoE: np.ndarray
    d_REV: np.ndarray
    rupture_start_idx: int
    rupture_end_idx: int


@dataclass
class RobustnessResult:
    spec: str
    episode: str
    window_start: str
    window_end: str
    state_spec: str
    proxy_set: str
    prob_delta: float
    prob_F: float
    prob_C: float
    status: str
    m_star: int
    rupture_start_idx: int
    rupture_end_idx: int
    n_divergences: int
    max_rhat: float
    min_ess_bulk: float
    trace_file: str
    posterior_csv: str


# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------
def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()



def find_column(columns: Sequence[str], aliases: Sequence[str], purpose: str) -> str:
    lookup = {str(c).strip(): c for c in columns}
    lower_lookup = {str(c).strip().lower(): c for c in columns}

    for alias in aliases:
        if alias in lookup:
            return lookup[alias]
        if alias.lower() in lower_lookup:
            return lower_lookup[alias.lower()]

    raise ValueError(
        f"Required observed series for '{purpose}' was not found. "
        f"Expected one of {list(aliases)}; available columns are {list(columns)}."
    )



def standardize(values: np.ndarray, base_n: int = 3) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float64)
    if arr.ndim != 1:
        raise ValueError("standardize() expects a one-dimensional array.")
    if len(arr) < max(3, base_n):
        raise ValueError("At least three observations are required for standardisation.")
    base = arr[:base_n]
    base_mean = float(np.nanmean(base))
    base_std = float(np.nanstd(base))
    if not np.isfinite(base_std) or base_std <= 1e-12:
        raise ValueError("Baseline standard deviation is zero or undefined.")
    return ((arr - base_mean) / base_std).astype(np.float64)



def ensure_numeric_no_missing(series: pd.Series, purpose: str) -> pd.Series:
    out = pd.to_numeric(series, errors="coerce")
    if out.isna().any():
        missing_idx = out.index[out.isna()].tolist()[:10]
        raise ValueError(
            f"Observed series '{purpose}' contains missing/non-numeric values at rows {missing_idx}."
        )
    return out.astype(float)



def make_relative_placebo_dummies(T: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    d_MB = np.zeros(T, dtype=np.float64)
    d_BoE = np.zeros(T, dtype=np.float64)
    d_REV = np.zeros(T, dtype=np.float64)

    mb_lo, mb_hi = PLACEBO_MB_SLICE
    boe_lo, boe_hi = PLACEBO_BOE_SLICE
    d_MB[mb_lo:min(mb_hi, T)] = 1.0
    d_BoE[boe_lo:min(boe_hi, T)] = 1.0
    if PLACEBO_REV_START < T:
        d_REV[PLACEBO_REV_START:] = 1.0
    return d_MB, d_BoE, d_REV



def infer_rupture_indices_from_dates(dates: pd.Series) -> Tuple[int, int]:
    rupture_mask = (dates >= BOE_START) & (dates <= pd.Timestamp("2022-09-30"))
    idx = np.flatnonzero(rupture_mask.to_numpy())
    if len(idx) < 3:
        raise ValueError(
            "The selected window does not contain the required rupture days "
            "(2022-09-28 to 2022-09-30)."
        )
    return int(idx[0]), int(idx[-1])



def prepare_frame(
    episode: EpisodeFrame,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    spec_name: str,
    state_spec: str,
    proxy_set: str,
    placebo_relative_timing: bool,
) -> PreparedData:
    df = episode.df.copy()
    mask = (df["date"] >= start_date) & (df["date"] <= end_date)
    sub = df.loc[mask].copy().reset_index(drop=True)
    if sub.empty:
        raise ValueError(
            f"Window for spec '{spec_name}' is empty after date filtering: "
            f"{start_date.date()} to {end_date.date()}."
        )

    fx = ensure_numeric_no_missing(sub["fx"], "FX proxy")
    reset = ensure_numeric_no_missing(sub["reset"], "reset/intervention proxy")

    if proxy_set == "main":
        gilt = ensure_numeric_no_missing(sub["gilt_main"], "main gilt proxy")
        liq = ensure_numeric_no_missing(sub["liq_main"], "main liquidity proxy")
    elif proxy_set == "alt":
        if "gilt_alt" not in sub.columns or "liq_alt" not in sub.columns:
            raise ValueError(
                f"Spec '{spec_name}' requires alternative observed proxies, but one or both "
                "alternative series are missing."
            )
        gilt = ensure_numeric_no_missing(sub["gilt_alt"], "alternative gilt proxy")
        liq = ensure_numeric_no_missing(sub["liq_alt"], "alternative liquidity proxy")
    else:
        raise ValueError(f"Unknown proxy_set: {proxy_set}")

    z_fx = standardize(fx.to_numpy()) * -1.0
    z_gilt = standardize(gilt.to_numpy())
    z_liq = standardize(liq.to_numpy())
    z_reset = standardize(reset.to_numpy())

    if placebo_relative_timing:
        d_MB, d_BoE, d_REV = make_relative_placebo_dummies(len(sub))
        r0, r1 = PLACEBO_RUPTURE_SLICE
        if len(sub) < r1:
            raise ValueError(
                f"Placebo spec '{spec_name}' contains too few observations for the rupture template."
            )
        rupture_start_idx, rupture_end_idx = r0, r1 - 1
    else:
        d_MB = ((sub["date"] >= MB_START) & (sub["date"] <= MB_END)).astype(np.float64).to_numpy()
        d_BoE = ((sub["date"] >= BOE_START) & (sub["date"] <= BOE_END)).astype(np.float64).to_numpy()
        d_REV = (sub["date"] >= REV_START).astype(np.float64).to_numpy()
        rupture_start_idx, rupture_end_idx = infer_rupture_indices_from_dates(sub["date"])

    return PreparedData(
        spec_name=spec_name,
        episode_name=episode.name,
        start_date=str(start_date.date()),
        end_date=str(end_date.date()),
        state_spec=state_spec,
        proxy_set=proxy_set,
        T=len(sub),
        dates=[d.strftime("%Y-%m-%d") for d in sub["date"]],
        z_fx=z_fx,
        z_gilt=z_gilt,
        z_liq=z_liq,
        z_reset=z_reset,
        d_MB=d_MB,
        d_BoE=d_BoE,
        d_REV=d_REV,
        rupture_start_idx=rupture_start_idx,
        rupture_end_idx=rupture_end_idx,
    )



def normalise_episode_file(
    path: Path,
    name: str,
    require_alt: bool,
) -> EpisodeFrame:
    if not path.exists():
        raise FileNotFoundError(f"Episode CSV not found: {path}")

    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"Episode CSV is empty: {path}")

    date_col = find_column(df.columns, DATE_ALIASES, "date")
    fx_col = find_column(df.columns, FX_ALIASES, "FX proxy")
    reset_col = find_column(df.columns, RESET_ALIASES, "reset proxy")
    gilt_main_col = find_column(df.columns, GILT_MAIN_ALIASES, "main gilt proxy")
    liq_main_col = find_column(df.columns, LIQ_MAIN_ALIASES, "main liquidity proxy")

    out = pd.DataFrame({
        "date": pd.to_datetime(df[date_col], errors="coerce"),
        "fx": pd.to_numeric(df[fx_col], errors="coerce"),
        "reset": pd.to_numeric(df[reset_col], errors="coerce"),
        "gilt_main": pd.to_numeric(df[gilt_main_col], errors="coerce"),
        "liq_main": pd.to_numeric(df[liq_main_col], errors="coerce"),
    })

    if require_alt:
        galt = find_column(df.columns, GILT_ALT_ALIASES, "alternative gilt proxy")
        lalt = find_column(df.columns, LIQ_ALT_ALIASES, "alternative liquidity proxy")
        out["gilt_alt"] = pd.to_numeric(df[galt], errors="coerce")
        out["liq_alt"] = pd.to_numeric(df[lalt], errors="coerce")

    if out["date"].isna().any():
        raise ValueError(f"Invalid dates detected in {path}")

    # No silent imputation: completed-results mode requires fully observed input series.
    missing_cols = [c for c in out.columns if out[c].isna().any()]
    if missing_cols:
        raise ValueError(
            f"Completed-results mode forbids missing values. Please repair {path}. "
            f"Missing values detected in columns {missing_cols}."
        )

    out = out.sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)
    return EpisodeFrame(name=name, df=out, source_path=path)


# -----------------------------------------------------------------------------
# Model
# -----------------------------------------------------------------------------
def build_latent_process(
    name: str,
    T: int,
    sigma: pm.Distribution,
    init_sigma: float,
    state_spec: str,
):
    if state_spec == "AR":
        rho = pm.Uniform(f"rho_{name}", lower=0.80, upper=0.995)
        eps = pm.Normal(f"eps_{name}", mu=0.0, sigma=1.0, shape=T)

        def step(e_t, x_tm1, rho_param, sigma_param):
            return rho_param * x_tm1 + sigma_param * e_t

        series, _ = pm.scan(
            fn=step,
            sequences=[eps[1:]],
            outputs_info=[eps[0] * sigma],
            non_sequences=[rho, sigma],
        )
        x = pt.concatenate([[eps[0] * sigma], series])
        return x

    if state_spec == "RW":
        init = pm.Normal(f"init_{name}", mu=0.0, sigma=init_sigma)
        steps = pm.Normal(f"steps_{name}", mu=0.0, sigma=sigma, shape=T - 1)
        x = pt.concatenate([[init], init + pt.cumsum(steps)])
        return x

    if state_spec == "LL":
        # Local-level state: level evolves as a random walk, but with a tighter innovation prior.
        init = pm.Normal(f"init_{name}", mu=0.0, sigma=init_sigma)
        level_sigma = pm.HalfNormal(f"level_sigma_{name}", sigma=0.25)
        steps = pm.Normal(f"steps_{name}", mu=0.0, sigma=level_sigma, shape=T - 1)
        x = pt.concatenate([[init], init + pt.cumsum(steps)])
        return x

    raise ValueError(f"Unknown state specification: {state_spec}")



def run_mcmc_estimation(
    data: PreparedData,
    draws: int,
    tune: int,
    chains: int,
    target_accept: float,
    seed: int,
) -> az.InferenceData:
    T = data.T
    coords = {"time": np.arange(T)}

    with pm.Model(coords=coords) as model:
        alpha = pm.Normal("alpha", mu=0.0, sigma=0.5, shape=4)

        beta_11 = pm.HalfNormal("beta_11", sigma=1.0)
        beta_12 = pm.HalfNormal("beta_12", sigma=1.0)
        beta_13 = pm.HalfNormal("beta_13", sigma=1.0)

        beta_21 = pm.HalfNormal("beta_21", sigma=1.0)
        beta_22 = pm.HalfNormal("beta_22", sigma=1.0)
        beta_23 = pm.HalfNormal("beta_23", sigma=1.0)

        beta_31 = pm.HalfNormal("beta_31", sigma=1.0)
        beta_32 = pm.HalfNormal("beta_32", sigma=1.0)

        beta_41 = pm.Normal("beta_41", mu=0.0, sigma=1.0)
        beta_42 = pm.Normal("beta_42", mu=0.0, sigma=1.0)

        sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.3, shape=4)
        sigma_states = pm.HalfNormal("sigma_states", sigma=0.5, shape=5)

        r_raw = build_latent_process("r_raw", T, sigma_states[0], 0.5, data.state_spec)
        log_pi = build_latent_process("log_pi", T, sigma_states[1], 0.5, data.state_spec)
        log_g = build_latent_process("log_g", T, sigma_states[2], 0.5, data.state_spec)
        tilde_eta = build_latent_process("tilde_eta", T, sigma_states[3], 0.5, data.state_spec)
        log_tau = build_latent_process("log_tau", T, sigma_states[4], 0.5, data.state_spec)

        r_t = pm.Deterministic("r_t", r_raw, dims=("time",))
        pi_t = pm.Deterministic("pi_t", pt.exp(log_pi), dims=("time",))
        g_t = pm.Deterministic("g_t", pt.exp(log_g), dims=("time",))
        eta_t = pm.Deterministic("eta_t", pt.exp(tilde_eta), dims=("time",))
        tau_t = pm.Deterministic("tau_t", pt.exp(log_tau), dims=("time",))

        d_BoE = pt.as_tensor_variable(data.d_BoE)
        d_REV = pt.as_tensor_variable(data.d_REV)

        mu_gilt = alpha[0] + beta_11 * pi_t + beta_12 * eta_t + beta_13 * tau_t
        mu_fx = alpha[1] + beta_21 * pi_t - beta_22 * g_t + beta_23 * tau_t
        mu_liq = alpha[2] + beta_31 * eta_t + beta_32 * tau_t
        mu_reset = alpha[3] + beta_41 * d_BoE + beta_42 * d_REV

        pm.Normal("obs_gilt", mu=mu_gilt, sigma=sigma_obs[0], observed=data.z_gilt, dims=("time",))
        pm.Normal("obs_fx", mu=mu_fx, sigma=sigma_obs[1], observed=data.z_fx, dims=("time",))
        pm.Normal("obs_liq", mu=mu_liq, sigma=sigma_obs[2], observed=data.z_liq, dims=("time",))
        pm.Normal("obs_reset", mu=mu_reset, sigma=sigma_obs[3], observed=data.z_reset, dims=("time",))

        delta_t = pm.Deterministic("delta_t", pi_t + eta_t * (pi_t ** 2) - g_t, dims=("time",))
        f1_t = pm.Deterministic("f1_t", r_t - pi_t, dims=("time",))
        f2_t = pm.Deterministic("f2_t", g_t - r_t - eta_t * (r_t ** 2), dims=("time",))
        f3_t = pm.Deterministic("f3_t", r_t + pi_t, dims=("time",))
        C_t = pm.Deterministic("C_t", f2_t + f1_t + eta_t * f1_t * f3_t, dims=("time",))

        trace = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=seed,
            progressbar=False,
            return_inferencedata=True,
        )

    return trace


# -----------------------------------------------------------------------------
# Evaluation and outputs
# -----------------------------------------------------------------------------
def evaluate_decision_rule(
    trace: az.InferenceData,
    data: PreparedData,
    m_star: int,
    trace_file: str,
    posterior_csv: str,
) -> Tuple[RobustnessResult, pd.DataFrame]:
    post = trace.posterior

    delta_samples = post["delta_t"].stack(sample=("chain", "draw")).values
    f1_samples = post["f1_t"].stack(sample=("chain", "draw")).values
    f2_samples = post["f2_t"].stack(sample=("chain", "draw")).values
    C_samples = post["C_t"].stack(sample=("chain", "draw")).values

    prob_delta_pos = np.mean(delta_samples > 0, axis=1)
    prob_C_neg = np.mean(C_samples < 0, axis=1)
    prob_F = np.mean((f1_samples > 0) & (f2_samples > 0), axis=1)

    r0 = data.rupture_start_idx
    r1 = data.rupture_end_idx

    rupture_prob_delta = prob_delta_pos[r0:r1 + 1]
    rupture_prob_F = prob_F[r0:r1 + 1]
    rupture_prob_C = prob_C_neg[r0:r1 + 1]

    pass_flag = False
    if len(rupture_prob_delta) >= m_star:
        for i in range(len(rupture_prob_delta) - m_star + 1):
            if (
                np.all(rupture_prob_delta[i:i + m_star] >= 0.95)
                and np.all(rupture_prob_F[i:i + m_star] <= 0.05)
                and np.all(rupture_prob_C[i:i + m_star] >= 0.95)
            ):
                pass_flag = True
                break

    div = 0
    if "sample_stats" in trace and "diverging" in trace.sample_stats:
        div = int(trace.sample_stats["diverging"].sum().values)

    rhat_ds = az.rhat(trace, method="rank")
    ess_ds = az.ess(trace, method="bulk")

    def ds_max(ds: az.data.inference_data.InferenceData | xr.Dataset) -> float:  # type: ignore[name-defined]
        vals: List[float] = []
        for v in ds.data_vars.values():
            arr = np.asarray(v.values).astype(float)
            arr = arr[np.isfinite(arr)]
            if arr.size:
                vals.append(float(arr.max()))
        return max(vals) if vals else float("nan")

    def ds_min(ds: az.data.inference_data.InferenceData | xr.Dataset) -> float:  # type: ignore[name-defined]
        vals: List[float] = []
        for v in ds.data_vars.values():
            arr = np.asarray(v.values).astype(float)
            arr = arr[np.isfinite(arr)]
            if arr.size:
                vals.append(float(arr.min()))
        return min(vals) if vals else float("nan")

    max_rhat = ds_max(rhat_ds)
    min_ess_bulk = ds_min(ess_ds)

    daily = pd.DataFrame(
        {
            "date": data.dates,
            "Pr_delta_gt_0": prob_delta_pos,
            "p_F": prob_F,
            "Pr_C_lt_0": prob_C_neg,
            "in_rupture_window": [int(r0 <= i <= r1) for i in range(data.T)],
        }
    )

    result = RobustnessResult(
        spec=data.spec_name,
        episode=data.episode_name,
        window_start=data.start_date,
        window_end=data.end_date,
        state_spec=data.state_spec,
        proxy_set=data.proxy_set,
        prob_delta=float(np.max(rupture_prob_delta)),
        prob_F=float(np.min(rupture_prob_F)),
        prob_C=float(np.max(rupture_prob_C)),
        status="Pass" if pass_flag else "Fail",
        m_star=m_star,
        rupture_start_idx=r0,
        rupture_end_idx=r1,
        n_divergences=div,
        max_rhat=float(max_rhat),
        min_ess_bulk=float(min_ess_bulk),
        trace_file=trace_file,
        posterior_csv=posterior_csv,
    )
    return result, daily



def spec_slug(name: str) -> str:
    out = name.lower()
    for a, b in [
        (" ", "_"),
        (",", ""),
        ("(", ""),
        (")", ""),
        ("/", "_"),
        ("-", "_"),
    ]:
        out = out.replace(a, b)
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")



def write_table_a1(results: Sequence[RobustnessResult], outpath: Path) -> None:
    lines = [
        r"¥begin{table}[t]",
        r"¥centering",
        r"¥caption{Completed robustness matrix for the theorem-analogue empirical test. This table reports posterior probabilities computed directly from script-estimated latent-state models. A ``Pass'' indicates that the conditions $¥Pr(¥delta_t>0¥mid Y_{1:T})¥ge 0.95$, $p_t^{F}¥le 0.05$, and $¥Pr(C_t<0¥mid Y_{1:T})¥ge 0.95$ are strictly satisfied for at least $m_¥ast=3$ consecutive days within the designated rupture window.}",
        r"¥label{tab:completed_robustness_matrix}",
        r"¥small",
        r"¥begin{tabular}{l c c c c}",
        r"¥hline",
        r"Specification & $¥Pr(¥delta_t>0¥mid Y_{1:T})$ & $p_t^{F}$ & $¥Pr(C_t<0¥mid Y_{1:T})$ & Status ¥¥",
        r"¥hline",
    ]
    for r in results:
        lines.append(
            f"{r.spec} & {r.prob_delta:.3f} & {r.prob_F:.3f} & {r.prob_C:.3f} & {r.status} ¥¥¥¥"  # noqa: E501
        )
    lines.extend([r"¥hline", r"¥end{tabular}", r"¥end{table}"])
    outpath.write_text("¥n".join(lines), encoding="utf-8")



def write_manifest(
    outpath: Path,
    args: argparse.Namespace,
    inputs: Sequence[Path],
    outputs: Sequence[Path],
    results: Sequence[RobustnessResult],
) -> None:
    payload = {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "pymc": pm.__version__,
            "arviz": az.__version__,
        },
        "arguments": {
            "baseline_csv": str(args.baseline_csv),
            "placebo_march_csv": str(args.placebo_march_csv),
            "placebo_aug_csv": str(args.placebo_aug_csv),
            "outdir": str(args.outdir),
            "draws": args.draws,
            "tune": args.tune,
            "chains": args.chains,
            "target_accept": args.target_accept,
            "seed": args.seed,
            "m_star": args.m_star,
        },
        "input_hashes": {str(p): sha256_of_file(p) for p in inputs},
        "output_hashes": {str(p): sha256_of_file(p) for p in outputs if p.exists()},
        "results": [asdict(r) for r in results],
    }
    outpath.write_text(json.dumps(payload, indent=2), encoding="utf-8")


# -----------------------------------------------------------------------------
# CLI
# -----------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Completed-results robustness estimation script.")
    p.add_argument("--baseline-csv", type=Path, required=True)
    p.add_argument("--placebo-march-csv", type=Path, required=True)
    p.add_argument("--placebo-aug-csv", type=Path, required=True)
    p.add_argument("--outdir", type=Path, default=Path("output_completed"))
    p.add_argument("--draws", type=int, default=1500)
    p.add_argument("--tune", type=int, default=1500)
    p.add_argument("--chains", type=int, default=2)
    p.add_argument("--target-accept", type=float, default=0.97)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--m-star", type=int, default=3)
    return p.parse_args()



def main() -> None:
    args = parse_args()
    outdir = args.outdir
    outdir.mkdir(parents=True, exist_ok=True)

    baseline = normalise_episode_file(args.baseline_csv, "Truss baseline episode", require_alt=True)
    placebo_march = normalise_episode_file(args.placebo_march_csv, "Placebo March 2022", require_alt=False)
    placebo_aug = normalise_episode_file(args.placebo_aug_csv, "Placebo August 2022", require_alt=False)

    spec_plan = [
        {
            "episode": baseline,
            "spec_name": "Baseline window, baseline proxy, AR state",
            "start": BASELINE_START,
            "end": BASELINE_END,
            "state_spec": "AR",
            "proxy_set": "main",
            "placebo_relative_timing": False,
        },
        {
            "episode": baseline,
            "spec_name": "Short window, baseline proxy, AR state",
            "start": SHORT_START,
            "end": SHORT_END,
            "state_spec": "AR",
            "proxy_set": "main",
            "placebo_relative_timing": False,
        },
        {
            "episode": baseline,
            "spec_name": "Long window, baseline proxy, AR state",
            "start": LONG_START,
            "end": LONG_END,
            "state_spec": "AR",
            "proxy_set": "main",
            "placebo_relative_timing": False,
        },
        {
            "episode": baseline,
            "spec_name": "Baseline window, alternative proxy set, AR state",
            "start": BASELINE_START,
            "end": BASELINE_END,
            "state_spec": "AR",
            "proxy_set": "alt",
            "placebo_relative_timing": False,
        },
        {
            "episode": baseline,
            "spec_name": "Baseline window, baseline proxy, local-level state",
            "start": BASELINE_START,
            "end": BASELINE_END,
            "state_spec": "LL",
            "proxy_set": "main",
            "placebo_relative_timing": False,
        },
        {
            "episode": baseline,
            "spec_name": "Baseline window, baseline proxy, random-walk state",
            "start": BASELINE_START,
            "end": BASELINE_END,
            "state_spec": "RW",
            "proxy_set": "main",
            "placebo_relative_timing": False,
        },
        {
            "episode": placebo_march,
            "spec_name": "Placebo window 1 (March 2022)",
            "start": placebo_march.df["date"].min(),
            "end": placebo_march.df["date"].max(),
            "state_spec": "AR",
            "proxy_set": "main",
            "placebo_relative_timing": True,
        },
        {
            "episode": placebo_aug,
            "spec_name": "Placebo window 2 (August 2022)",
            "start": placebo_aug.df["date"].min(),
            "end": placebo_aug.df["date"].max(),
            "state_spec": "AR",
            "proxy_set": "main",
            "placebo_relative_timing": True,
        },
    ]

    results: List[RobustnessResult] = []
    written_outputs: List[Path] = []

    for idx, cfg in enumerate(spec_plan, start=1):
        print(f"[{idx}/{len(spec_plan)}] Estimating: {cfg['spec_name']}")
        prepared = prepare_frame(
            episode=cfg["episode"],
            start_date=cfg["start"],
            end_date=cfg["end"],
            spec_name=cfg["spec_name"],
            state_spec=cfg["state_spec"],
            proxy_set=cfg["proxy_set"],
            placebo_relative_timing=cfg["placebo_relative_timing"],
        )

        trace = run_mcmc_estimation(
            prepared,
            draws=args.draws,
            tune=args.tune,
            chains=args.chains,
            target_accept=args.target_accept,
            seed=args.seed,
        )

        slug = spec_slug(cfg["spec_name"])
        trace_path = outdir / f"trace_{slug}.nc"
        posterior_path = outdir / f"posterior_daily_{slug}.csv"

        az.to_netcdf(trace, trace_path)
        result, daily = evaluate_decision_rule(
            trace,
            prepared,
            m_star=args.m_star,
            trace_file=trace_path.name,
            posterior_csv=posterior_path.name,
        )
        daily.to_csv(posterior_path, index=False)

        results.append(result)
        written_outputs.extend([trace_path, posterior_path])

    if len(results) != 8:
        raise RuntimeError(
            f"Expected 8 completed robustness rows, but obtained {len(results)}."
        )

    summary_csv = outdir / "posterior_summary_statistics.csv"
    diagnostics_csv = outdir / "mcmc_diagnostics_summary.csv"
    table_tex = outdir / "tableA1_completed_robustness_matrix.tex"
    manifest_json = outdir / "reproducibility_manifest_completed.json"

    results_df = pd.DataFrame([asdict(r) for r in results])
    results_df.to_csv(summary_csv, index=False)
    results_df[
        [
            "spec",
            "n_divergences",
            "max_rhat",
            "min_ess_bulk",
            "trace_file",
            "posterior_csv",
        ]
    ].to_csv(diagnostics_csv, index=False)
    write_table_a1(results, table_tex)

    written_outputs.extend([summary_csv, diagnostics_csv, table_tex])
    write_manifest(
        manifest_json,
        args,
        inputs=[args.baseline_csv, args.placebo_march_csv, args.placebo_aug_csv],
        outputs=written_outputs,
        results=results,
    )

    print("=" * 72)
    print("Completed-results robustness estimation finished successfully.")
    print(f"Output directory: {outdir.resolve()}")
    print("Created files:")
    for p in [summary_csv, diagnostics_csv, table_tex, manifest_json]:
        print(f"  - {p.name}")
    print("=" * 72)


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] --baseline-csv BASELINE_CSV
                                --placebo-march-csv PLACEBO_MARCH_CSV
                                --placebo-aug-csv PLACEBO_AUG_CSV
                                [--outdir OUTDIR] [--draws DRAWS]
                                [--tune TUNE] [--chains CHAINS]
                                [--target-accept TARGET_ACCEPT] [--seed SEED]
                                [--m-star M_STAR]
colab_kernel_launcher.py: error: the following arguments are required: --baseline-csv, --placebo-march-csv, --placebo-aug-csv
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --baseline-csv, --placebo-march-csv, --placebo-aug-csv

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_642/2525492397.py", line 919, in <cell line: 0>
    main()
  File "/tmp/ipykernel_642/2525492397.py", line 752, in main
    args = parse_args()
           ^^^^^^^^^^^^
  File "/tmp/ipyk

TypeError: object of type 'NoneType' has no len()

In [2]:
#!/usr/bin/env python3
"""
Colab direct-run empirical analogue script matched to the user's current /content files.

This version is tailored to the file names visible in the current Colab workspace:
    /content/truss_fx_merged.csv
    /content/U.S.-Dollars_to_U.K-Pound_Sterling _Spot_Exchange_Rate_2023.xlsx
    /content/truss_daily.csv
    /content/truss_daily_full.csv

Design choice
-------------
The currently uploaded files provide:
- a merged Truss-episode daily FX + BoE-purchase dataset; and
- a FRED daily USD/GBP history workbook.

They do NOT provide independent daily long-gilt-yield and bid-ask-spread series in a
clean merged form for every robustness row. Accordingly, this script is "path-matched and
runnable" with the present files, but it remains econometrically conservative: the main and
alternative proxy sets are built from observed FX levels / returns when independent columns
are unavailable.

So this script is suitable for:
- direct Colab execution without argparse;
- producing a script-generated Appendix A.1 table;
- replacing yfinance dependence with local uploaded files.

It is NOT, by itself, a warrant for a strong "fully completed robustness results" claim.
"""

from __future__ import annotations

import hashlib
import json
import logging
import os
import platform
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

os.environ.setdefault("PYTENSOR_FLAGS", "optimizer=fast_compile")

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

logging.getLogger("pymc").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------------------------------------------------------------
# Colab paths matched to the current uploaded filenames
# -----------------------------------------------------------------------------
CONFIG: Dict[str, object] = {
    "baseline_csv": "/content/truss_fx_merged.csv",
    "fx_history_xlsx": "/content/U.S.-Dollars_to_U.K-Pound_Sterling _Spot_Exchange_Rate_2023.xlsx",
    "boe_daily_csv": "/content/truss_daily_full.csv",
    "outdir": "/content/output_empirical_currentfiles",
    "draws": 1000,
    "tune": 1000,
    "chains": 2,
    "target_accept": 0.99,
    "seed": 42,
}

MB_START = pd.Timestamp("2022-09-23")
MB_END = pd.Timestamp("2022-09-27")
BOE_START = pd.Timestamp("2022-09-28")
BOE_END = pd.Timestamp("2022-10-14")
REV_START = pd.Timestamp("2022-10-15")

BASELINE_START = pd.Timestamp("2022-09-20")
BASELINE_END = pd.Timestamp("2022-10-31")
SHORT_START = pd.Timestamp("2022-09-23")
SHORT_END = pd.Timestamp("2022-10-14")
LONG_START = pd.Timestamp("2022-09-13")
LONG_END = pd.Timestamp("2022-10-31")

PLACEBO_MB_SLICE = (3, 8)
PLACEBO_BOE_SLICE = (8, 25)
PLACEBO_REV_START = 25
PLACEBO_RUPTURE_SLICE = (8, 10)  # inclusive in evaluation


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def standardize(series: np.ndarray, base_start: int = 0, base_end: int = 3) -> np.ndarray:
    s = np.asarray(series, dtype=np.float64)
    mu = np.nanmean(s[base_start:base_end])
    sd = np.nanstd(s[base_start:base_end])
    return ((s - mu) / (sd + 1e-6)).astype(np.float64)


def clean_baseline_frame(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    if "date" not in df.columns:
        raise ValueError("baseline_csv must contain a 'date' column")
    if "usd_per_gbp" not in df.columns:
        raise ValueError("baseline_csv must contain 'usd_per_gbp'")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").copy()
    df = df[(df["date"] >= LONG_START) & (df["date"] <= BASELINE_END)].copy()
    df["usd_per_gbp"] = pd.to_numeric(df["usd_per_gbp"], errors="coerce").ffill()

    reset_col = None
    for cand in ["boe_purchase_total", "boe_purchase", "boe_purchases_mn"]:
        if cand in df.columns:
            reset_col = cand
            break
    if reset_col is None:
        raise ValueError("baseline_csv must contain one of boe_purchase_total / boe_purchase / boe_purchases_mn")
    df["reset_proxy"] = pd.to_numeric(df[reset_col], errors="coerce").fillna(0.0)

    # If independent market proxies exist, use them. Otherwise use observable FX-based transforms.
    if "gilt_yield" in df.columns:
        df["gilt_main"] = pd.to_numeric(df["gilt_yield"], errors="coerce").ffill()
    else:
        df["gilt_main"] = -df["usd_per_gbp"]

    if "bid_ask_spread" in df.columns:
        df["liq_main"] = pd.to_numeric(df["bid_ask_spread"], errors="coerce").ffill()
    else:
        df["liq_main"] = -df["usd_per_gbp"]

    # Alternative proxy set from observable FX transforms.
    log_fx = np.log(df["usd_per_gbp"].clip(lower=1e-8))
    fx_ret = log_fx.diff().fillna(0.0)
    df["gilt_alt"] = fx_ret.abs()
    df["liq_alt"] = fx_ret.rolling(3, min_periods=1).std().fillna(0.0)
    return df


def clean_placebo_frame(fx_history_xlsx: Path, start: str, end: str, name: str) -> pd.DataFrame:
    fx = pd.read_excel(fx_history_xlsx, sheet_name="Daily")
    fx.columns = [str(c).strip() for c in fx.columns]
    date_col = "observation_date" if "observation_date" in fx.columns else fx.columns[0]
    value_col = "DEXUSUK" if "DEXUSUK" in fx.columns else fx.columns[1]
    fx = fx[[date_col, value_col]].copy()
    fx.columns = ["date", "usd_per_gbp"]
    fx["date"] = pd.to_datetime(fx["date"])
    fx["usd_per_gbp"] = pd.to_numeric(fx["usd_per_gbp"], errors="coerce")
    fx = fx[(fx["date"] >= pd.Timestamp(start)) & (fx["date"] <= pd.Timestamp(end))].copy()
    fx = fx.dropna(subset=["usd_per_gbp"]).sort_values("date").head(42).copy()
    if len(fx) < 30:
        raise ValueError(f"{name}: too few FX observations after slicing ({len(fx)})")

    fx["reset_proxy"] = 0.0
    fx["gilt_main"] = -fx["usd_per_gbp"]
    fx["liq_main"] = -fx["usd_per_gbp"]
    log_fx = np.log(fx["usd_per_gbp"].clip(lower=1e-8))
    fx_ret = log_fx.diff().fillna(0.0)
    fx["gilt_alt"] = fx_ret.abs()
    fx["liq_alt"] = fx_ret.rolling(3, min_periods=1).std().fillna(0.0)
    return fx


def make_relative_placebo_dummies(T: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    d_MB = np.zeros(T, dtype=np.float64)
    d_BoE = np.zeros(T, dtype=np.float64)
    d_REV = np.zeros(T, dtype=np.float64)
    a, b = PLACEBO_MB_SLICE
    c, d = PLACEBO_BOE_SLICE
    d_MB[a:b] = 1.0
    d_BoE[c:d] = 1.0
    if PLACEBO_REV_START < T:
        d_REV[PLACEBO_REV_START:] = 1.0
    return d_MB, d_BoE, d_REV


def prepare_episode(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp,
                    proxy_set: str, state_spec: str, spec_name: str,
                    placebo_relative_timing: bool = False) -> Dict[str, object]:
    sub = df[(df["date"] >= start) & (df["date"] <= end)].copy()
    sub = sub.sort_values("date").reset_index(drop=True)
    if sub.empty:
        raise ValueError(f"No data for specification: {spec_name}")

    if proxy_set == "baseline":
        gilt_series = sub["gilt_main"].to_numpy(np.float64)
        liq_series = sub["liq_main"].to_numpy(np.float64)
    elif proxy_set == "alternative":
        gilt_series = sub["gilt_alt"].to_numpy(np.float64)
        liq_series = sub["liq_alt"].to_numpy(np.float64)
    else:
        raise ValueError(f"Unknown proxy_set: {proxy_set}")

    z_fx = standardize(sub["usd_per_gbp"].to_numpy(np.float64)) * -1.0
    z_gilt = standardize(gilt_series)
    z_liq = standardize(liq_series)
    z_reset = standardize(sub["reset_proxy"].to_numpy(np.float64))

    if placebo_relative_timing:
        d_MB, d_BoE, d_REV = make_relative_placebo_dummies(len(sub))
        rupture_start_idx, rupture_end_idx = PLACEBO_RUPTURE_SLICE
    else:
        d_MB = ((sub["date"] >= MB_START) & (sub["date"] <= MB_END)).astype(np.float64).to_numpy()
        d_BoE = ((sub["date"] >= BOE_START) & (sub["date"] <= BOE_END)).astype(np.float64).to_numpy()
        d_REV = (sub["date"] >= REV_START).astype(np.float64).to_numpy()
        rupture_mask = ((sub["date"] >= BOE_START) & (sub["date"] <= pd.Timestamp("2022-09-30"))).to_numpy()
        idx = np.where(rupture_mask)[0]
        if len(idx) == 0:
            raise ValueError(f"No rupture-window observations found for {spec_name}")
        rupture_start_idx, rupture_end_idx = int(idx[0]), int(idx[-1])

    return {
        "spec_name": spec_name,
        "state_spec": state_spec,
        "proxy_set": proxy_set,
        "dates": sub["date"].dt.strftime("%Y-%m-%d").tolist(),
        "z_fx": z_fx,
        "z_gilt": z_gilt,
        "z_liq": z_liq,
        "z_reset": z_reset,
        "d_MB": d_MB,
        "d_BoE": d_BoE,
        "d_REV": d_REV,
        "rupture_start_idx": rupture_start_idx,
        "rupture_end_idx": rupture_end_idx,
        "T": len(sub),
    }


def build_model(data: Dict[str, object], draws: int, tune: int, chains: int,
                target_accept: float, seed: int):
    z_gilt = data["z_gilt"]
    z_fx = data["z_fx"]
    z_liq = data["z_liq"]
    z_reset = data["z_reset"]
    d_MB = data["d_MB"]
    d_BoE = data["d_BoE"]
    d_REV = data["d_REV"]
    T = data["T"]
    state_spec = data["state_spec"]

    with pm.Model() as model:
        alpha = pm.Normal("alpha", mu=0, sigma=0.5, shape=4)
        beta_11 = pm.HalfNormal("beta_11", sigma=1)
        beta_12 = pm.HalfNormal("beta_12", sigma=1)
        beta_13 = pm.HalfNormal("beta_13", sigma=1)
        beta_21 = pm.HalfNormal("beta_21", sigma=1)
        beta_22 = pm.HalfNormal("beta_22", sigma=1)
        beta_23 = pm.HalfNormal("beta_23", sigma=1)
        beta_31 = pm.HalfNormal("beta_31", sigma=1)
        beta_32 = pm.HalfNormal("beta_32", sigma=1)
        beta_41 = pm.Normal("beta_41", mu=0, sigma=1)
        beta_42 = pm.Normal("beta_42", mu=0, sigma=1)
        sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.2, shape=4)

        if state_spec == "AR":
            rhos = pm.Uniform("rhos", lower=0.8, upper=0.99, shape=5)
            sigma_states = pm.HalfNormal("sigma_states", sigma=0.1, shape=5)
            init_dist = pm.Normal.dist(mu=0, sigma=0.5)
            r_raw = pm.AR("r_raw", rho=rhos[0], sigma=sigma_states[0], shape=T, init_dist=init_dist)
            log_pi = pm.AR("log_pi", rho=rhos[1], sigma=sigma_states[1], shape=T, init_dist=init_dist)
            log_g = pm.AR("log_g", rho=rhos[2], sigma=sigma_states[2], shape=T, init_dist=init_dist)
            tilde_eta = pm.AR("tilde_eta", rho=rhos[3], sigma=sigma_states[3], shape=T, init_dist=init_dist)
            log_tau = pm.AR("log_tau", rho=rhos[4], sigma=sigma_states[4], shape=T, init_dist=init_dist)
        elif state_spec == "RW":
            sigma_states = pm.HalfNormal("sigma_states", sigma=0.1, shape=5)
            r_raw = pm.GaussianRandomWalk("r_raw", sigma=sigma_states[0], shape=T)
            log_pi = pm.GaussianRandomWalk("log_pi", sigma=sigma_states[1], shape=T)
            log_g = pm.GaussianRandomWalk("log_g", sigma=sigma_states[2], shape=T)
            tilde_eta = pm.GaussianRandomWalk("tilde_eta", sigma=sigma_states[3], shape=T)
            log_tau = pm.GaussianRandomWalk("log_tau", sigma=sigma_states[4], shape=T)
        elif state_spec == "LL":
            sigma_level = pm.HalfNormal("sigma_level", sigma=0.1, shape=5)
            level_r = pm.GaussianRandomWalk("level_r", sigma=sigma_level[0], shape=T)
            level_pi = pm.GaussianRandomWalk("level_pi", sigma=sigma_level[1], shape=T)
            level_g = pm.GaussianRandomWalk("level_g", sigma=sigma_level[2], shape=T)
            level_eta = pm.GaussianRandomWalk("level_eta", sigma=sigma_level[3], shape=T)
            level_tau = pm.GaussianRandomWalk("level_tau", sigma=sigma_level[4], shape=T)
            r_raw = level_r
            log_pi = level_pi
            log_g = level_g
            tilde_eta = level_eta
            log_tau = level_tau
        else:
            raise ValueError(f"Unknown state_spec: {state_spec}")

        r_t = pm.Deterministic("r_t", r_raw)
        pi_t = pm.Deterministic("pi_t", pt.exp(log_pi))
        g_t = pm.Deterministic("g_t", pt.exp(log_g))
        eta_t = pm.Deterministic("eta_t", pt.exp(tilde_eta))
        tau_t = pm.Deterministic("tau_t", pt.exp(log_tau))

        mu_gilt = alpha[0] + beta_11 * pi_t + beta_12 * eta_t + beta_13 * tau_t
        mu_fx = alpha[1] + beta_21 * pi_t - beta_22 * g_t + beta_23 * tau_t
        mu_liq = alpha[2] + beta_31 * eta_t + beta_32 * tau_t
        mu_reset = alpha[3] + beta_41 * d_BoE + beta_42 * d_REV

        pm.Normal("obs_gilt", mu=mu_gilt, sigma=sigma_obs[0], observed=z_gilt)
        pm.Normal("obs_fx", mu=mu_fx, sigma=sigma_obs[1], observed=z_fx)
        pm.Normal("obs_liq", mu=mu_liq, sigma=sigma_obs[2], observed=z_liq)
        pm.Normal("obs_reset", mu=mu_reset, sigma=sigma_obs[3], observed=z_reset)

        delta_t = pm.Deterministic("delta_t", pi_t + eta_t * (pi_t**2) - g_t)
        f1_t = pm.Deterministic("f1_t", r_t - pi_t)
        f2_t = pm.Deterministic("f2_t", g_t - r_t - eta_t * (r_t**2))
        f3_t = pm.Deterministic("f3_t", r_t + pi_t)
        C_t = pm.Deterministic("C_t", f2_t + f1_t + eta_t * f1_t * f3_t)

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            random_seed=seed,
            target_accept=target_accept,
            progressbar=False,
            compute_convergence_checks=False,
            return_inferencedata=True,
        )

    return idata


def evaluate_rule(idata, rupture_start_idx: int, rupture_end_idx: int) -> Tuple[float, float, float, str]:
    delta_samples = idata.posterior["delta_t"].stack(sample=("chain", "draw")).values
    f1_samples = idata.posterior["f1_t"].stack(sample=("chain", "draw")).values
    f2_samples = idata.posterior["f2_t"].stack(sample=("chain", "draw")).values
    C_samples = idata.posterior["C_t"].stack(sample=("chain", "draw")).values

    prob_delta_pos = np.mean(delta_samples > 0, axis=1)
    prob_C_neg = np.mean(C_samples < 0, axis=1)
    prob_F = np.mean((f1_samples > 0) & (f2_samples > 0), axis=1)

    sl = slice(rupture_start_idx, rupture_end_idx + 1)
    rupture_prob_delta = prob_delta_pos[sl]
    rupture_prob_F = prob_F[sl]
    rupture_prob_C = prob_C_neg[sl]

    m_star = 3
    passed = False
    if len(rupture_prob_delta) >= m_star:
        for i in range(len(rupture_prob_delta) - m_star + 1):
            c1 = np.all(rupture_prob_delta[i : i + m_star] >= 0.95)
            c2 = np.all(rupture_prob_F[i : i + m_star] <= 0.05)
            c3 = np.all(rupture_prob_C[i : i + m_star] >= 0.95)
            if c1 and c2 and c3:
                passed = True
                break

    repr_prob_delta = float(np.max(rupture_prob_delta)) if len(rupture_prob_delta) else 0.0
    repr_prob_F = float(np.min(rupture_prob_F)) if len(rupture_prob_F) else 0.0
    repr_prob_C = float(np.max(rupture_prob_C)) if len(rupture_prob_C) else 0.0
    return repr_prob_delta, repr_prob_F, repr_prob_C, ("Pass" if passed else "Fail")


def posterior_daily_csv(idata, dates: List[str], out_csv: Path) -> None:
    def _mean(name: str) -> np.ndarray:
        arr = idata.posterior[name].mean(dim=("chain", "draw")).values
        return np.asarray(arr, dtype=float)

    out = pd.DataFrame({
        "date": dates,
        "delta_mean": _mean("delta_t"),
        "f1_mean": _mean("f1_t"),
        "f2_mean": _mean("f2_t"),
        "C_mean": _mean("C_t"),
        "r_mean": _mean("r_t"),
        "pi_mean": _mean("pi_t"),
        "g_mean": _mean("g_t"),
        "eta_mean": _mean("eta_t"),
        "tau_mean": _mean("tau_t"),
    })
    out.to_csv(out_csv, index=False)


def diagnostics(idata) -> Tuple[float, float]:
    summ = az.summary(idata, round_to=6)
    max_rhat = float(np.nanmax(summ["r_hat"].to_numpy())) if "r_hat" in summ else float("nan")
    min_ess = float(np.nanmin(summ["ess_bulk"].to_numpy())) if "ess_bulk" in summ else float("nan")
    return max_rhat, min_ess


def write_table(results: List[Dict[str, object]], outpath: Path) -> None:
    lines = [
        r"¥begin{table}[t]",
        r"¥centering",
        r"¥caption{Script-generated robustness matrix using the currently uploaded local files.}",
        r"¥label{tab:completed_robustness_matrix_currentfiles}",
        r"¥small",
        r"¥begin{tabular}{l c c c c}",
        r"¥hline",
        r"Specification & $¥Pr(¥delta_t>0¥mid Y_{1:T})$ & $p_t^{F}$ & $¥Pr(C_t<0¥mid Y_{1:T})$ & Status ¥¥",
        r"¥hline",
    ]
    for r in results:
        lines.append(
            f"{r['spec']} & {r['prob_delta']:.3f} & {r['prob_F']:.3f} & {r['prob_C']:.3f} & {r['status']} ¥¥")
    lines.extend([r"¥hline", r"¥end{tabular}", r"¥end{table}"])
    outpath.write_text("¥n".join(lines), encoding="utf-8")


def main() -> None:
    cfg = dict(CONFIG)
    baseline_csv = Path(str(cfg["baseline_csv"]))
    fx_history_xlsx = Path(str(cfg["fx_history_xlsx"]))
    outdir = Path(str(cfg["outdir"]))
    ensure_dir(outdir)

    if not baseline_csv.exists():
        raise FileNotFoundError(f"Missing baseline file: {baseline_csv}")
    if not fx_history_xlsx.exists():
        raise FileNotFoundError(f"Missing FX history file: {fx_history_xlsx}")

    print("[1/4] Loading current local files...")
    baseline_df = clean_baseline_frame(baseline_csv)
    placebo_march_df = clean_placebo_frame(fx_history_xlsx, "2022-03-01", "2022-04-30", "Placebo March 2022")
    placebo_aug_df = clean_placebo_frame(fx_history_xlsx, "2022-08-01", "2022-09-20", "Placebo August 2022")

    specs = [
        ("Baseline window, baseline proxy, AR state", baseline_df, BASELINE_START, BASELINE_END, "baseline", "AR", False),
        ("Short window, baseline proxy, AR state", baseline_df, SHORT_START, SHORT_END, "baseline", "AR", False),
        ("Long window, baseline proxy, AR state", baseline_df, LONG_START, LONG_END, "baseline", "AR", False),
        ("Baseline window, alternative proxy set, AR state", baseline_df, BASELINE_START, BASELINE_END, "alternative", "AR", False),
        ("Baseline window, baseline proxy, local-level state", baseline_df, BASELINE_START, BASELINE_END, "baseline", "LL", False),
        ("Baseline window, baseline proxy, random-walk state", baseline_df, BASELINE_START, BASELINE_END, "baseline", "RW", False),
        ("Placebo window 1 (March 2022)", placebo_march_df, placebo_march_df["date"].min(), placebo_march_df["date"].max(), "baseline", "AR", True),
        ("Placebo window 2 (August 2022)", placebo_aug_df, placebo_aug_df["date"].min(), placebo_aug_df["date"].max(), "baseline", "AR", True),
    ]

    results: List[Dict[str, object]] = []
    manifest: Dict[str, object] = {
        "python": platform.python_version(),
        "packages": {
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "pymc": pm.__version__,
            "arviz": az.__version__,
        },
        "inputs": {
            str(baseline_csv): sha256_of_file(baseline_csv),
            str(fx_history_xlsx): sha256_of_file(fx_history_xlsx),
        },
        "outputs": [],
        "note": "Current-files Colab version. Where independent daily gilt/liquidity series are unavailable, FX-based observable transforms are used.",
    }

    print("[2/4] Estimating all eight A.1 rows...")
    for idx, (spec_name, frame, start, end, proxy_set, state_spec, placebo_rel) in enumerate(specs, start=1):
        print(f"  - ({idx}/8) {spec_name}")
        data = prepare_episode(frame, start, end, proxy_set, state_spec, spec_name, placebo_relative_timing=placebo_rel)
        idata = build_model(
            data,
            draws=int(cfg["draws"]),
            tune=int(cfg["tune"]),
            chains=int(cfg["chains"]),
            target_accept=float(cfg["target_accept"]),
            seed=int(cfg["seed"]),
        )
        prob_delta, prob_F, prob_C, status = evaluate_rule(idata, data["rupture_start_idx"], data["rupture_end_idx"])

        safe = f"spec_{idx:02d}"
        trace_path = outdir / f"{safe}_trace.nc"
        posterior_csv = outdir / f"{safe}_posterior_daily.csv"
        idata.to_netcdf(trace_path)
        posterior_daily_csv(idata, data["dates"], posterior_csv)
        max_rhat, min_ess = diagnostics(idata)

        results.append({
            "spec": spec_name,
            "prob_delta": prob_delta,
            "prob_F": prob_F,
            "prob_C": prob_C,
            "status": status,
            "max_rhat": max_rhat,
            "min_ess_bulk": min_ess,
            "trace_file": trace_path.name,
            "posterior_csv": posterior_csv.name,
        })
        manifest["outputs"].extend([
            {"path": str(trace_path), "sha256": sha256_of_file(trace_path)},
            {"path": str(posterior_csv), "sha256": sha256_of_file(posterior_csv)},
        ])

    print("[3/4] Writing output tables and manifest...")
    results_csv = outdir / "robustness_results_currentfiles.csv"
    pd.DataFrame(results).to_csv(results_csv, index=False)
    table_tex = outdir / "tableA1_currentfiles_completed_matrix.tex"
    write_table(results, table_tex)
    manifest["outputs"].append({"path": str(results_csv), "sha256": sha256_of_file(results_csv)})
    manifest["outputs"].append({"path": str(table_tex), "sha256": sha256_of_file(table_tex)})
    manifest_path = outdir / "manifest_currentfiles.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print("[4/4] Done.")
    print(f"Outputs written to: {outdir}")
    print(f"Main LaTeX table:   {table_tex}")
    print(f"Results CSV:        {results_csv}")
    print(f"Manifest:           {manifest_path}")


if __name__ == "__main__":
    main()


[1/4] Loading current local files...
[2/4] Estimating all eight A.1 rows...
  - (1/8) Baseline window, baseline proxy, AR state
  - (2/8) Short window, baseline proxy, AR state
  - (3/8) Long window, baseline proxy, AR state
  - (4/8) Baseline window, alternative proxy set, AR state
  - (5/8) Baseline window, baseline proxy, local-level state


ERROR:pytensor.graph.rewriting.basic:Rewrite failure due to: local_subtensor_merge
ERROR:pytensor.graph.rewriting.basic:node: Subtensor{i}(Subtensor{start:stop}.0, 0)
ERROR:pytensor.graph.rewriting.basic:TRACEBACK:
ERROR:pytensor.graph.rewriting.basic:Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytensor/graph/rewriting/basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytensor/graph/rewriting/basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytensor/tensor/rewriting/subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "/usr/local/lib/python3.12/dist-packages/pytensor/tensor/rewriting/subtensor.py", line 869, in merge_two_slices
    n_val = sl1.stop - 1 - sl2 * sl1.step
            ~~~~~~~~~~~~~^~~~~~~~~

  - (6/8) Baseline window, baseline proxy, random-walk state


ERROR:pytensor.graph.rewriting.basic:Rewrite failure due to: local_subtensor_merge
ERROR:pytensor.graph.rewriting.basic:node: Subtensor{i}(Subtensor{start:stop}.0, 0)
ERROR:pytensor.graph.rewriting.basic:TRACEBACK:
ERROR:pytensor.graph.rewriting.basic:Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pytensor/graph/rewriting/basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytensor/graph/rewriting/basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pytensor/tensor/rewriting/subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "/usr/local/lib/python3.12/dist-packages/pytensor/tensor/rewriting/subtensor.py", line 869, in merge_two_slices
    n_val = sl1.stop - 1 - sl2 * sl1.step
            ~~~~~~~~~~~~~^~~~~~~~~

  - (7/8) Placebo window 1 (March 2022)
  - (8/8) Placebo window 2 (August 2022)
[3/4] Writing output tables and manifest...
[4/4] Done.
Outputs written to: /content/output_empirical_currentfiles
Main LaTeX table:   /content/output_empirical_currentfiles/tableA1_currentfiles_completed_matrix.tex
Results CSV:        /content/output_empirical_currentfiles/robustness_results_currentfiles.csv
Manifest:           /content/output_empirical_currentfiles/manifest_currentfiles.json


In [3]:
import pandas as pd
import glob

for f in sorted(glob.glob('/content/output_empirical_currentfiles/spec_*_posterior_daily.csv')):
    df = pd.read_csv(f)
    print("\nFILE:", f)
    print(df.columns.tolist())
    print(df.head())


FILE: /content/output_empirical_currentfiles/spec_01_posterior_daily.csv
['date', 'delta_mean', 'f1_mean', 'f2_mean', 'C_mean', 'r_mean', 'pi_mean', 'g_mean', 'eta_mean', 'tau_mean']
         date  delta_mean   f1_mean   f2_mean      C_mean    r_mean   pi_mean  \
0  2022-09-20   15.893768 -3.130615  0.710189  -15.893768  0.001619  3.132234   
1  2022-09-21   16.848117 -3.065832  0.406182  -16.848117  0.036602  3.102434   
2  2022-09-22   31.035595 -3.568982  0.229235  -31.035595  0.097188  3.666170   
3  2022-09-23  141.798185 -5.259959  0.132624 -141.798185  0.329968  5.589927   
4  2022-09-24  143.563999 -5.057647  0.118162 -143.563999  0.359169  5.416815   

     g_mean  eta_mean  tau_mean  
0  1.105887  1.183464  2.161904  
1  1.056913  1.545494  1.976972  
2  1.021338  2.127834  2.225550  
3  0.930371  3.152695  3.274098  
4  0.857152  3.360680  3.125396  

FILE: /content/output_empirical_currentfiles/spec_02_posterior_daily.csv
['date', 'delta_mean', 'f1_mean', 'f2_mean', 'C_mea

In [5]:
import os
import glob
import numpy as np
import pandas as pd
import arviz as az

OUTDIR = "/content/output_empirical_currentfiles"

def recompute_from_trace(trace_path, posterior_csv_path, spec_id):
    idata = az.from_netcdf(trace_path)
    daily = pd.read_csv(posterior_csv_path)

    # posterior draws
    delta = idata.posterior["delta_t"].stack(sample=("chain", "draw")).values
    f1    = idata.posterior["f1_t"].stack(sample=("chain", "draw")).values
    f2    = idata.posterior["f2_t"].stack(sample=("chain", "draw")).values
    C     = idata.posterior["C_t"].stack(sample=("chain", "draw")).values

    # daily posterior probabilities
    prob_delta = np.mean(delta > 0, axis=1)
    prob_F     = np.mean((f1 > 0) & (f2 > 0), axis=1)
    prob_C     = np.mean(C < 0, axis=1)

    # rupture window
    if spec_id in [7, 8]:
        # placebo relative timing in currentfiles script
        rs, re = 8, 10
    else:
        # actual Truss rupture window
        mask = (pd.to_datetime(daily["date"]) >= pd.Timestamp("2022-09-28")) & \
               (pd.to_datetime(daily["date"]) <= pd.Timestamp("2022-09-30"))
        idx = np.where(mask)[0]
        if len(idx) == 0:
            raise ValueError(f"No rupture dates found in {posterior_csv_path}")
        rs, re = int(idx[0]), int(idx[-1])

    rpd = prob_delta[rs:re+1]
    rpf = prob_F[rs:re+1]
    rpc = prob_C[rs:re+1]

    passed = False
    m_star = 3
    if len(rpd) >= m_star:
        for i in range(len(rpd) - m_star + 1):
            c1 = np.all(rpd[i:i+m_star] >= 0.95)
            c2 = np.all(rpf[i:i+m_star] <= 0.05)
            c3 = np.all(rpc[i:i+m_star] >= 0.95)
            if c1 and c2 and c3:
                passed = True
                break

    out = {
        "spec_id": spec_id,
        "rupture_dates": daily.loc[rs:re, "date"].tolist(),
        "prob_delta": float(np.max(rpd)) if len(rpd) else np.nan,
        "prob_F": float(np.min(rpf)) if len(rpf) else np.nan,
        "prob_C": float(np.max(rpc)) if len(rpc) else np.nan,
        "status_recomputed": "Pass" if passed else "Fail",
    }
    return out

rows = []
for trace_path in sorted(glob.glob(os.path.join(OUTDIR, "spec_*_trace.nc"))):
    name = os.path.basename(trace_path)
    spec_id = int(name.split("_")[1])
    posterior_csv_path = os.path.join(OUTDIR, f"spec_{spec_id:02d}_posterior_daily.csv")
    rows.append(recompute_from_trace(trace_path, posterior_csv_path, spec_id))

recomputed = pd.DataFrame(rows).sort_values("spec_id")
display(recomputed)

# 元の results CSV と比較
orig = pd.read_csv(os.path.join(OUTDIR, "robustness_results_currentfiles.csv"))
orig = orig.copy()
orig["spec_id"] = range(1, len(orig) + 1)

check = orig.merge(recomputed, on="spec_id", how="left", suffixes=("_orig", "_re"))

display(
    check[[
        "spec_id", "spec",
        "prob_delta_orig", "prob_F_orig", "prob_C_orig", "status",
        "prob_delta_re", "prob_F_re", "prob_C_re", "status_recomputed",
        "rupture_dates"
    ]].rename(columns={
        "status": "status_orig"
    })
)

,spec_id,rupture_dates,prob_delta,prob_F,prob_C,status_recomputed
0,1,"[2022-09-28, 2022-09-29, 2022-09-30]",1.0000,0.0000,1.0000,Pass
1,2,"[2022-09-28, 2022-09-29, 2022-09-30]",1.0000,0.0000,1.0000,Fail
2,3,"[2022-09-28, 2022-09-29, 2022-09-30]",1.0000,0.0000,1.0000,Pass
3,4,"[2022-09-28, 2022-09-29, 2022-09-30]",0.9925,0.0005,0.9925,Pass
4,5,"[2022-09-28, 2022-09-29, 2022-09-30]",1.0000,0.0000,1.0000,Pass
5,6,"[2022-09-28, 2022-09-29, 2022-09-30]",1.0000,0.0000,1.0000,Pass
6,7,"[2022-03-11, 2022-03-14, 2022-03-15]",1.0000,0.0000,1.0000,Pass
7,8,"[2022-08-11, 2022-08-12, 2022-08-15]",1.0000,0.0000,1.0000,Fail


,spec_id,spec,prob_delta_orig,prob_F_orig,prob_C_orig,status_orig,prob_delta_re,prob_F_re,prob_C_re,status_recomputed,rupture_dates
0,1,"Baseline window, baseline proxy, AR state",1.0000,0.0000,1.0000,Pass,1.0000,0.0000,1.0000,Pass,"[2022-09-28, 2022-09-29, 2022-09-30]"
1,2,"Short window, baseline proxy, AR state",1.0000,0.0000,1.0000,Fail,1.0000,0.0000,1.0000,Fail,"[2022-09-28, 2022-09-29, 2022-09-30]"
2,3,"Long window, baseline proxy, AR state",1.0000,0.0000,1.0000,Pass,1.0000,0.0000,1.0000,Pass,"[2022-09-28, 2022-09-29, 2022-09-30]"
3,4,"Baseline window, alternative proxy set, AR state",0.9925,0.0005,0.9925,Pass,0.9925,0.0005,0.9925,Pass,"[2022-09-28, 2022-09-29, 2022-09-30]"
4,5,"Baseline window, baseline proxy, local-level s...",1.0000,0.0000,1.0000,Pass,1.0000,0.0000,1.0000,Pass,"[2022-09-28, 2022-09-29, 2022-09-30]"
5,6,"Baseline window, baseline proxy, random-walk s...",1.0000,0.0000,1.0000,Pass,1.0000,0.0000,1.0000,Pass,"[2022-09-28, 2022-09-29, 2022-09-30]"
6,7,Placebo window 1 (March 2022),1.0000,0.0000,1.0000,Pass,1.0000,0.0000,1.0000,Pass,"[2022-03-11, 2022-03-14, 2022-03-15]"
7,8,Placebo window 2 (August 2022),1.0000,0.0000,1.0000,Fail,1.0000,0.0000,1.0000,Fail,"[2022-08-11, 2022-08-12, 2022-08-15]"
